# SSMP-OneShot: 1-Shot Semantic Segmentation Recreation
# "Adaptive Self-Supporting Prototype Learning for Remote Sensing Few-Shot Semantic Segmentation"

**Paper:** Weihao Shen, Ailong Ma, Junjue Wang, Zhuo Zheng, Yanfei Zhong,
*"Adaptive Self-Supporting Prototype Learning for Remote Sensing Few-Shot Semantic Segmentation"*,
IEEE Transactions on Geoscience and Remote Sensing, vol. 62, 2024.
DOI: `10.1109/TGRS.2024.3435086`

---

## 1. What this notebook is

This notebook is an end-to-end recreation of the **1-Shot (1-way 1-shot)** setting of the **SSMP** framework for remote sensing semantic segmentation on **iSAID-5i**.

### The 1-Shot Workflow in SSMP:
```text
Support Image + Support Mask (1 exemplar)       Query Image
          |                                          |
          v                                          v
   ResNet-50 + Projection                     ResNet-50 + Projection
          |                                          |
          v                                          v
Support Feature Map (F_s)                     Query Feature Map (F_q)
          |                                          |
          v                                          |
Masked Average Pooling (MAP) -> Prototype P_supp     |
          |                                          |
          v                                          |
HPR (Hyperprototype Clustering on F_s foreground)   |
          |                                          |
          v                                          v
Guided Prototype Matching (GPM) <--------------------+
          |
          v
Initial Query Prediction
          |
          v
SSM (Self-Supporting Query Prototype Refinement)
          |
          v
ASBP (Adaptive Self-Supporting Background Prototype from Query)
          |
          v
Refine Stage -> Final 1-Shot Prediction (Evaluated via mIoU / FB-IoU)
```

### Key Differences from Zero-Shot:
- **Zero-Shot:** Class prototype comes from text descriptions via CLIP text encoder (no support image/mask).
- **1-Shot (This Notebook):** Class prototype is extracted directly from **1 real support image and its binary mask** using Masked Average Pooling (MAP), and hyperprototypes are clustered from the support foreground features.

## 2. Module-by-Module Architecture Mapping

| # | SSMP Paper Module | 1-Shot Implementation | Description |
|---|---|---|---|
| 1 | Support Exemplar | 1 support image + binary mask | Standard 1-shot episodic setting |
| 2 | Visual Prototype | Masked Average Pooling (MAP) | $\mathbf{P}_{supp} = \frac{\sum F_s \odot M_s}{\sum M_s}$ |
| 3 | Frozen Backbone | Pretrained ResNet-50 (layer3 + layer4) | Spatial features (3072 channels, stride 16) |
| 4 | Projection Head | 1x1 Conv + GroupNorm + ReLU + 1x1 Conv | Maps features to 512-dim embedding space |
| 5 | Prototype Adapter | Support Prototype Adapter MLP | Fine-tunes visual support prototype |
| 6 | HPR (HPC + GPM) | K-means clustering on support foreground | Produces $K_{hyper}=4$ hyperprototypes |
| 7 | SSM (Self-Support) | Query pseudo-mask MAP + hyperprototypes | Query guides query via learnable $\lambda$ fusion |
| 8 | ASBP (Background) | Affinity matrix on query background pixels | Adaptive per-pixel background prototype |
| 9 | Losses | $L = L_{seg} + \beta L_{pseudo}$ ($\beta=0.2$) | Weighted Cross-Entropy on cosine logits |
| 10 | Optimization | Adam + CosineAnnealingLR (25 epochs) | Stable convergence and improved model quality |

In [ ]:
# =============================================================================
# 3. INSTALL DEPENDENCIES (if running on Colab)
# =============================================================================
# On Google Colab, uncomment and run:
# !pip install -q kagglehub torch torchvision matplotlib scikit-learn pandas
print("Dependencies ready.")

In [ ]:
# =============================================================================
# 4. IMPORTS
# =============================================================================
import os
import sys
import math
import time
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import ResNet50_Weights

print("PyTorch Version :", torch.__version__)
print("Torchvision     :", torchvision.__version__)
print("CUDA Available  :", torch.cuda.is_available())

In [ ]:
# =============================================================================
# 5. CONFIGURATION - 1-SHOT SETTING
# =============================================================================
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------ task setup -----
N_SHOTS = 1              # 1-way 1-shot
IMG_SIZE = 256           # crops resized to 256 x 256
EMBED_DIM = 512          # embedding dimension for prototypes and pixel features
FEAT_SIZE = IMG_SIZE // 16   # 16 x 16 spatial feature map
SCORE_SCALE = 8.0        # cosine score temperature

# ------------------------------------------------------------- iSAID class --
ISAID_CLASSES = [
    "plane", "ship", "storage tank", "baseball diamond", "tennis court",
    "basketball court", "ground track field", "soccer ball field", "harbor",
    "bridge", "large vehicle", "small vehicle", "helicopter", "roundabout",
    "swimming pool",
]
CLASS_TO_ID = {name: i + 1 for i, name in enumerate(ISAID_CLASSES)}
ID_TO_CLASS = {i: name for name, i in CLASS_TO_ID.items()}

# Disjoint Base / Unseen class split (Fold 0 as novel classes)
UNSEEN_CLASSES = ["ship", "plane", "harbor", "bridge", "large vehicle"]
BASE_CLASSES = [c for c in ISAID_CLASSES if c not in UNSEEN_CLASSES]

BASE_IDS = sorted(CLASS_TO_ID[c] for c in BASE_CLASSES)
UNSEEN_IDS = sorted(CLASS_TO_ID[c] for c in UNSEEN_CLASSES)

assert set(BASE_IDS).isdisjoint(UNSEEN_IDS), "Base and Unseen must be disjoint"

# ------------------------------------------------------------ data paths ----
DATA_ROOT = Path("/content/data/iSAID5i")
DATA_ROOT_CANDIDATES = [
    Path("/content/data/iSAID5i"),
    Path("/content/iSAID5i"),
    Path("./data/iSAID5i"),
    Path("/content/drive/MyDrive/iSAID5i"),
]
ALLOW_SYNTHETIC_FALLBACK = True
MAX_SCAN_IMAGES = 4000

MAX_TRAIN_IMAGES = 500
MAX_VAL_IMAGES = 100
MAX_TEST_IMAGES = 50
ABLATION_IMAGES_PER_CLASS = 15

# ------------------------------------------------------------- training -----
# Increased epochs & quality improvements
EPOCHS = 25
BATCH_SIZE = 4
LR = 5e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
USE_PSEUDO_LOSS = True
PSEUDO_LOSS_WEIGHT = 0.2
FG_LOSS_WEIGHT = 3.0
NUM_WORKERS = 2

# ------------------------------------------------------- model / modules ----
NUM_HYPERPROTOS = 4
KMEANS_ITERS = 5
FG_THRESH = 0.6
BG_THRESH = 0.6

MODULES = {"hpr": True, "ssm": True, "asbp": True, "refine": True}

CKPT_DIR = Path("/content/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT = CKPT_DIR / "best_ssmp_oneshot.pt"
CLASS_COLOR_MAP = {}

print(f"Base Classes ({len(BASE_CLASSES)})   : {BASE_CLASSES}")
print(f"Unseen Classes ({len(UNSEEN_CLASSES)}) : {UNSEEN_CLASSES}")

## 6. Dataset Download / Loading - iSAID-5i

The notebook expects the iSAID-5i dataset layout:
```text
iSAID5i/
├── images/     *.png / *.jpg patches (256x256)
├── masks/      *.png ground truth masks (class ids 0..15)
└── splits/     train.txt, val.txt, test.txt
```
If real data is not yet downloaded, KaggleHub downloads it automatically below.

In [ ]:
# =============================================================================
# 6b. LOCATE DATASET + SYNTHETIC FALLBACK
# =============================================================================
import colorsys

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
IMG_DIR_NAMES = {"images", "img", "image", "jpegimages"}
MASK_DIR_NAMES = {"masks", "mask", "semantic_masks", "semanticmask", "gt", "labels"}

def find_dirs(root, wanted_names, max_depth=3):
    wanted = {w.lower() for w in wanted_names}
    found = {}
    root = Path(root)
    if not root.exists():
        return found
    for dirpath, dirnames, _ in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames.clear()
            continue
        name = Path(dirpath).name.lower()
        if name in wanted:
            found.setdefault(name, []).append(Path(dirpath))
    return found

def locate_dataset(candidates):
    for root in candidates:
        root = Path(root)
        imgs = find_dirs(root, IMG_DIR_NAMES)
        msks = find_dirs(root, MASK_DIR_NAMES)
        if imgs and msks:
            img_dir = sorted(imgs.values(), key=len)[0][0]
            mask_dir = sorted(msks.values(), key=len)[0][0]
            return {"root": root, "images": img_dir, "masks": mask_dir}
    return None

def _class_color(cid):
    hue = (cid * 0.61803398875) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
    return np.array([r * 255, g * 255, b * 255], dtype=np.float32)

def build_synthetic_dataset(root, n_train=60, n_val=20, n_test=20, size=IMG_SIZE, seed=SEED):
    rng = np.random.default_rng(seed)
    root = Path(root)
    img_dir, msk_dir, split_dir = root / "images", root / "masks", root / "splits"
    for d in (img_dir, msk_dir, split_dir):
        d.mkdir(parents=True, exist_ok=True)

    all_classes = BASE_CLASSES + UNSEEN_CLASSES
    yy, xx = np.ogrid[:size, :size]

    for split, n in (("train", n_train), ("val", n_val), ("test", n_test)):
        stems = []
        for i in range(n):
            stem = f"{split}_{i:03d}"
            img = rng.normal(110, 30, (size, size, 3)).clip(0, 255).astype(np.float32)
            for _ in range(4):
                cx, cy = rng.integers(0, size, 2)
                rr = int(rng.integers(size // 8, size // 3))
                blob = ((xx - cx) ** 2 + (yy - cy) ** 2) <= rr * rr
                col = _class_color(int(rng.integers(1, 16)))
                img[blob] = img[blob] * 0.5 + col * 0.5
            mask = np.zeros((size, size), dtype=np.uint8)

            for _ in range(int(rng.integers(1, 4))):
                cname = str(all_classes[rng.integers(len(all_classes))])
                cid = CLASS_TO_ID[cname]
                color = _class_color(cid)
                x0, y0 = rng.integers(8, max(9, size - 96), 2)
                w, h = rng.integers(24, 90, 2)
                x1, y1 = min(size - 1, x0 + w), min(size - 1, y0 + h)
                region = (xx >= x0) & (xx <= x1) & (yy >= y0) & (yy <= y1)
                img[region] = img[region] * 0.35 + color * 0.65
                mask[region] = cid

            Image.fromarray(img.astype(np.uint8)).save(img_dir / f"{stem}.png")
            Image.fromarray(mask).save(msk_dir / f"{stem}.png")
            stems.append(stem)
        (split_dir / f"{split}.txt").write_text("\n".join(stems) + "\n")

    return {"root": root, "images": img_dir, "masks": msk_dir}

dataset_info = locate_dataset(DATA_ROOT_CANDIDATES)
if dataset_info is None:
    if ALLOW_SYNTHETIC_FALLBACK:
        print("Real dataset not found in default paths - will check Kaggle or synthetic fallback.")
    else:
        raise FileNotFoundError("Real dataset not found.")

In [ ]:
# Optional automatic download via KaggleHub if running on Google Colab:
try:
    import kagglehub
    kaggle_path = kagglehub.dataset_download("abinaya2704/isaid-5i-dataset")
    print("Dataset found/downloaded to:", kaggle_path)
    DATA_ROOT = Path(kaggle_path)
    DATA_ROOT_CANDIDATES.insert(0, DATA_ROOT)
    dataset_info = locate_dataset(DATA_ROOT_CANDIDATES)
except Exception as e:
    print("Kaggle download skipped or not configured:", e)

if dataset_info is None and ALLOW_SYNTHETIC_FALLBACK:
    print("Generating synthetic placeholder dataset for debugging...")
    dataset_info = build_synthetic_dataset(DATA_ROOT)
    USING_SYNTHETIC = True
else:
    USING_SYNTHETIC = False

print("Dataset root :", dataset_info["root"])
print("Images dir   :", dataset_info["images"])
print("Masks dir    :", dataset_info["masks"])

In [ ]:
# =============================================================================
# 6c. IMAGE / MASK LOADING AND 1-SHOT EPISODIC DATASET
# =============================================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def load_mask(path, size=IMG_SIZE):
    img = Image.open(path)
    if img.mode in ("P", "L", "I", "I;16"):
        arr = np.array(img).astype(np.int32)
    else:
        rgb = np.array(img.convert("RGB")).astype(np.int32)
        grey = np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 0], rgb[..., 2])
        if grey:
            arr = rgb[..., 0]
        elif CLASS_COLOR_MAP:
            arr = np.zeros(rgb.shape[:2], dtype=np.int32)
            for (r, g, b), cid in CLASS_COLOR_MAP.items():
                arr[(rgb == np.array([r, g, b])).all(axis=-1)] = cid
        else:
            arr = rgb[..., 0]

    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.uint8)
    if arr.shape[0] != size or arr.shape[1] != size:
        arr = np.array(Image.fromarray(arr).resize((size, size), Image.NEAREST))
    return arr.astype(np.int64)

def load_image(path, size=IMG_SIZE):
    img = Image.open(path).convert("RGB")
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return img

def read_split_stems(info, split):
    split_dir = Path(info["root"]) / "splits"
    split_dir.mkdir(exist_ok=True)
    f = split_dir / f"{split}.txt"
    if f.exists():
        return [l.strip() for l in f.read_text().splitlines() if l.strip()]

    all_stems = sorted(p.stem for p in Path(info["images"]).glob("*") if p.suffix.lower() in IMG_EXTS)
    rng = random.Random(SEED)
    rng.shuffle(all_stems)
    n = len(all_stems)
    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))
    parts = {
        "train": all_stems[:n_train],
        "val": all_stems[n_train:n_train + n_val],
        "test": all_stems[n_train + n_val:] or all_stems[-1:],
    }
    for name, stems in parts.items():
        (split_dir / f"{name}.txt").write_text("\n".join(stems) + "\n")
    return parts[split]

class OneShotISaidDataset(Dataset):
    """1-Shot Episodic Dataset: returns (Support Image, Support Mask, Query Image, Query Mask)"""
    def __init__(self, info, split, allowed_ids, img_size=IMG_SIZE, max_images=None):
        self.images_dir = Path(info["images"])
        self.masks_dir = Path(info["masks"])
        self.img_size = img_size
        self.split = split
        self.allowed_ids = list(allowed_ids)

        stems = read_split_stems(info, split)
        self.samples = []
        self.present_ids = []
        self.class_to_indices = {cid: [] for cid in self.allowed_ids}

        scanned = 0
        for stem in stems:
            if max_images is not None and len(self.samples) >= max_images:
                break
            if scanned >= MAX_SCAN_IMAGES:
                break
            img_p = self._find_file(self.images_dir, stem)
            msk_p = self._find_file(self.masks_dir, stem)
            if img_p is None or msk_p is None:
                continue
            scanned += 1
            m = load_mask(msk_p, img_size)
            present = sorted({int(v) for v in np.unique(m) if v in self.allowed_ids})
            if present:
                idx = len(self.samples)
                self.samples.append((img_p, msk_p))
                self.present_ids.append(present)
                for cid in present:
                    self.class_to_indices[cid].append(idx)

        print(f"[{split.upper()}] Loaded {len(self.samples)} images across {len(self.allowed_ids)} classes.")

    @staticmethod
    def _find_file(folder, stem):
        for ext in IMG_EXTS:
            p = folder / f"{stem}{ext}"
            if p.exists():
                return p
        return None

    def __len__(self):
        return len(self.samples)

    def _load_sample_tensor(self, idx, target_cid, augment=False):
        img_p, msk_p = self.samples[idx]
        img = load_image(img_p, self.img_size)
        mask_arr = load_mask(msk_p, self.img_size)

        if augment:
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask_arr = np.ascontiguousarray(np.fliplr(mask_arr))
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
                mask_arr = np.ascontiguousarray(np.flipud(mask_arr))

        x = transforms.ToTensor()(img)
        x = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(x)
        binary_mask = torch.from_numpy(mask_arr == target_cid).long()
        return x, binary_mask

    def __getitem__(self, idx):
        # Sample target class from present classes in query image
        present = self.present_ids[idx]
        target_cid = random.choice(present)

        # Sample 1 support image containing target_cid (distinct if possible)
        candidates = self.class_to_indices[target_cid]
        other_candidates = [i for i in candidates if i != idx]
        supp_idx = random.choice(other_candidates) if other_candidates else idx

        is_train = (self.split == "train")
        q_img, q_mask = self._load_sample_tensor(idx, target_cid, augment=is_train)
        s_img, s_mask = self._load_sample_tensor(supp_idx, target_cid, augment=is_train)

        return {
            "query_image": q_img,
            "query_mask": q_mask,
            "supp_image": s_img,
            "supp_mask": s_mask,
            "target_class_id": target_cid,
            "target_class_name": ID_TO_CLASS[target_cid],
        }

train_set = OneShotISaidDataset(dataset_info, "train", BASE_IDS, max_images=MAX_TRAIN_IMAGES)
val_set = OneShotISaidDataset(dataset_info, "val", BASE_IDS, max_images=MAX_VAL_IMAGES)
test_set = OneShotISaidDataset(dataset_info, "test", UNSEEN_IDS, max_images=MAX_TEST_IMAGES)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# =============================================================================
# 7. VISUALIZE 1-SHOT EPISODES (Support Pair & Query Pair)
# =============================================================================
def denormalize(img_t):
    mean = torch.tensor(IMAGENET_MEAN, device=img_t.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=img_t.device).view(3, 1, 1)
    return (img_t * std + mean).clamp(0, 1)

def show_oneshot_episodes(dataset, n=3):
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, 4)

    for i in range(n):
        ep = dataset[i]
        cname = ep["target_class_name"]

        s_img = denormalize(ep["supp_image"]).permute(1, 2, 0).numpy()
        s_msk = ep["supp_mask"].numpy()
        q_img = denormalize(ep["query_image"]).permute(1, 2, 0).numpy()
        q_msk = ep["query_mask"].numpy()

        axes[i, 0].imshow(s_img)
        axes[i, 0].set_title(f"Support Image ({cname})")
        axes[i, 1].imshow(s_msk, cmap="gray")
        axes[i, 1].set_title(f"Support Mask ({cname})")

        axes[i, 2].imshow(q_img)
        axes[i, 2].set_title(f"Query Image ({cname})")
        axes[i, 3].imshow(q_msk, cmap="gray")
        axes[i, 3].set_title(f"Query Ground Truth")

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_oneshot_episodes(train_set, n=2)

In [ ]:
# =============================================================================
# 8. FROZEN RESNET-50 BACKBONE & PROJECTION HEAD
# =============================================================================
class ResNetFeatureExtractor(nn.Module):
    """Pretrained ResNet-50: extracts multi-level spatial feature map (3072 channels)."""
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        net = torchvision.models.resnet50(weights=weights)
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1 = net.layer1
        self.layer2 = net.layer2
        self.layer3 = net.layer3      # stride 16, 1024 channels
        self.layer4 = net.layer4      # stride 32, 2048 channels
        self.out_channels = 1024 + 2048
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        c1 = self.stem(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        f3 = self.layer3(c3)
        f4 = self.layer4(f3)
        f4 = F.interpolate(f4, size=f3.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f3, f4], dim=1)  # (B, 3072, H/16, W/16)

class ProjectionHead(nn.Module):
    """Maps ResNet features to shared 512-dim embedding space."""
    def __init__(self, in_channels, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=1),
            nn.GroupNorm(8, 512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, embed_dim, kernel_size=1),
        )

    def forward(self, f):
        z = self.net(f)
        return F.normalize(z, dim=1)

class SupportPrototypeAdapter(nn.Module):
    """Fine-tunes the extracted support visual prototype."""
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, proto):
        return F.normalize(proto + self.scale * self.mlp(proto), dim=-1)

In [ ]:
# =============================================================================
# 9. 1-SHOT SUPPORT PROTOTYPE EXTRACTION (Masked Average Pooling - MAP)
# =============================================================================
def extract_support_prototype_map(support_embeds, support_mask):
    """
    support_embeds : (HW, C) support pixel embeddings
    support_mask   : (HW,) binary mask at feature resolution
    returns        : (C,) normalized support prototype
    """
    fg_mask = support_mask.bool()
    if fg_mask.sum() == 0:
        # Fallback to mean embedding if support mask is empty at feature resolution
        return F.normalize(support_embeds.mean(dim=0), dim=0)
    fg_feats = support_embeds[fg_mask]
    proto = fg_feats.mean(dim=0)
    return F.normalize(proto, dim=0)

In [ ]:
# =============================================================================
# 10. HPR (HYPERPROTOTYPE CLUSTERING + GUIDED PROTOTYPE MATCHING)
# =============================================================================
def kmeans_prototypes(feats, k, coords=None, iters=KMEANS_ITERS, spatial_weight=0.5):
    """Lightweight HPC-inspired k-means in joint feature + spatial space."""
    M, C = feats.shape
    k = max(1, min(k, M))
    f = feats.detach()
    f_aug = torch.cat([f, coords.detach() * spatial_weight], dim=1) if coords is not None else f

    centre_idx = [int(torch.argmax(f_aug.norm(dim=1)))]
    d = torch.cdist(f_aug, f_aug[centre_idx]).squeeze(1)
    while len(centre_idx) < k:
        idx = int(torch.argmax(d))
        centre_idx.append(idx)
        d = torch.minimum(d, torch.cdist(f_aug, f_aug[[idx]]).squeeze(1))
    centres = f_aug[centre_idx]

    for _ in range(iters):
        assign = torch.cdist(f_aug, centres).argmin(dim=1)
        new_centres = []
        for j in range(k):
            m = (assign == j)
            new_centres.append(f_aug[m].mean(dim=0) if m.any() else centres[j])
        centres = torch.stack(new_centres)

    with torch.no_grad():
        d_final = torch.cdist(f_aug, centres)
        assign = d_final.argmin(dim=1)
        onehot = F.one_hot(assign, k).T.float()
        empty = (onehot.sum(dim=1) == 0)
        if empty.any():
            far = d_final.max(dim=0).indices
            for j in torch.nonzero(empty, as_tuple=False).squeeze(1):
                onehot[j] = 0.0
                onehot[j, far[j]] = 1.0
    counts = onehot.sum(dim=1, keepdim=True).clamp(min=1.0)
    centroids = (onehot @ feats) / counts
    return centroids

class SimplifiedHPR(nn.Module):
    """HPR: Clusters support foreground features into multiple hyperprototypes."""
    def __init__(self, num_hyperprotos=NUM_HYPERPROTOS, iters=KMEANS_ITERS, spatial_weight=0.5):
        super().__init__()
        self.k = num_hyperprotos
        self.iters = iters
        self.spatial_weight = spatial_weight

    def forward(self, pixel_embeds, base_proto, select_mask=None):
        hw, c = pixel_embeds.shape
        k = min(self.k, hw)
        if select_mask is None or select_mask.sum() < k:
            sim = pixel_embeds @ base_proto
            sel = torch.zeros(hw, dtype=torch.bool, device=pixel_embeds.device)
            sel[torch.topk(sim, max(k, int(0.25 * hw))).indices] = True
            select_mask = sel

        pts = pixel_embeds[select_mask]
        side = max(int(round(math.sqrt(hw))), 1)
        idx = torch.nonzero(select_mask, as_tuple=False).squeeze(1)
        yy = (idx // side).float() / max(side - 1, 1)
        xx = (idx % side).float() / max(side - 1, 1)
        coords = torch.stack([yy, xx], dim=1)

        hyper = kmeans_prototypes(pts, k, coords=coords, iters=self.iters, spatial_weight=self.spatial_weight)
        return F.normalize(hyper, dim=1)

def gpm_match(pixel_embeds, prototypes, temperature=1.0):
    """Guided Prototype Matching (GPM): cosine similarity + soft aggregation."""
    sim = pixel_embeds @ prototypes.t()
    weights = F.softmax(sim / temperature, dim=1)
    score = (weights * sim).sum(dim=1)
    assign = sim.argmax(dim=1)
    return score, assign, sim

def scores_to_logits(fg_score, bg_score, scale=SCORE_SCALE):
    return torch.stack([scale * bg_score, scale * fg_score], dim=-1)

In [ ]:
# =============================================================================
# 11. SSM (SELF-SUPPORTING MATCHING) & ASBP (ADAPTIVE BACKGROUND)
# =============================================================================
def confident_masks(logits, fg_thresh=FG_THRESH, bg_thresh=BG_THRESH):
    prob = F.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)
    fg_conf = (prob[:, 1] >= fg_thresh) & (pred == 1)
    bg_conf = (prob[:, 0] >= bg_thresh) & (pred == 0)
    return fg_conf, bg_conf, prob

def masked_average(pixel_embeds, mask):
    if mask.dtype != torch.bool:
        mask = mask.bool()
    if int(mask.sum()) == 0:
        return None
    return F.normalize(pixel_embeds[mask].mean(dim=0), dim=0)

def asbp_background(pixel_embeds, bg_mask, max_bg_pixels=512):
    """Adaptive Self-Supporting Background Prototype (affinity matrix)."""
    hw, c = pixel_embeds.shape
    idx = torch.nonzero(bg_mask.bool(), as_tuple=False).squeeze(1)
    if idx.numel() == 0:
        proto = F.normalize(pixel_embeds.mean(dim=0), dim=0)
        return proto.unsqueeze(0).expand(hw, -1), (pixel_embeds @ proto)
    if idx.numel() > max_bg_pixels:
        pick = torch.linspace(0, idx.numel() - 1, max_bg_pixels, device=idx.device).long()
        idx = idx[pick]

    fb = pixel_embeds[idx]
    aff = fb @ pixel_embeds.t()
    w = F.softmax(aff, dim=0)
    bg_proto_map = F.normalize(w.t() @ fb, dim=1)
    bg_score = (pixel_embeds * bg_proto_map).sum(dim=1)
    return bg_proto_map, bg_score

def global_background(pixel_embeds, fg_score, frac=0.5):
    hw = pixel_embeds.shape[0]
    n = max(1, int(hw * frac))
    sel = torch.topk(fg_score, n, largest=False).indices
    mask = torch.zeros(hw, dtype=torch.bool, device=pixel_embeds.device)
    mask[sel] = True
    proto = F.normalize(pixel_embeds[mask].mean(dim=0), dim=0)
    return proto, (pixel_embeds @ proto)

In [ ]:
# =============================================================================
# 12. COMPLETE 1-SHOT SSMP MODEL
# =============================================================================
class OneShotSSMP(nn.Module):
    def __init__(self, modules=None, embed_dim=EMBED_DIM):
        super().__init__()
        self.backbone = ResNetFeatureExtractor(pretrained=True)
        self.project = ProjectionHead(self.backbone.out_channels, embed_dim)
        self.proto_adapter = SupportPrototypeAdapter(embed_dim)
        self.hpr = SimplifiedHPR()
        self.log_lambda = nn.Parameter(torch.zeros(2))
        self.flags = dict(MODULES if modules is None else modules)

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def extract_embeddings(self, images):
        with torch.no_grad():
            feats = self.backbone(images)
        z = self.project(feats)
        b, c, h, w = z.shape
        embeds = z.permute(0, 2, 3, 1).reshape(b, h * w, c)
        return embeds, (h, w)

    @staticmethod
    def _fuse(set_a, set_b, lam):
        k = max(set_a.shape[0], set_b.shape[0])
        if set_a.shape[0] < k:
            set_a = torch.cat([set_a, set_a[-1:].repeat(k - set_a.shape[0], 1)], dim=0)
        if set_b.shape[0] < k:
            set_b = torch.cat([set_b, set_b[-1:].repeat(k - set_b.shape[0], 1)], dim=0)
        return F.normalize(lam[0] * set_a + lam[1] * set_b, dim=1)

    def _hyperprototypes(self, emb, base, select_mask=None):
        if self.flags["hpr"]:
            extra = self.hpr(emb, base, select_mask=select_mask)
            return torch.cat([base.unsqueeze(0), extra], dim=0)
        if select_mask is None:
            return base.unsqueeze(0)
        q = masked_average(emb, select_mask)
        return (q if q is not None else base).unsqueeze(0)

    def _background_score(self, emb, bg_conf, bg_proto0):
        if self.flags["asbp"]:
            _, bg_score = asbp_background(emb, bg_conf)
            return bg_score
        bg_p = masked_average(emb, bg_conf)
        return emb @ (bg_p if bg_p is not None else bg_proto0)

    def forward(self, query_images, supp_images, supp_masks):
        """
        query_images : (B, 3, H, W)
        supp_images  : (B, 3, H, W)
        supp_masks   : (B, H, W) binary masks
        """
        q_embeds, (fh, fw) = self.extract_embeddings(query_images)
        s_embeds, _ = self.extract_embeddings(supp_images)
        lam = F.softmax(self.log_lambda, dim=0)

        # Downsample support masks to feature resolution
        s_masks_feat = F.interpolate(supp_masks.float().unsqueeze(1), size=(fh, fw),
                                     mode="nearest").squeeze(1).reshape(query_images.shape[0], -1)

        final_logits, init_logits, fg_scores, assigns, proto_sets = [], [], [], [], []

        for i in range(query_images.shape[0]):
            q_emb = q_embeds[i]
            s_emb = s_embeds[i]
            s_mask = s_masks_feat[i]

            # 1. Support Visual Prototype via MAP + Adapter
            raw_supp_proto = extract_support_prototype_map(s_emb, s_mask)
            base_proto = self.proto_adapter(raw_supp_proto)

            # 2. HPR on support foreground (or query candidate) + GPM
            supp_hyper = self._hyperprototypes(s_emb, base_proto, select_mask=(s_mask == 1))
            fg0, assign0, _ = gpm_match(q_emb, supp_hyper)
            bg_proto0, bg_score0 = global_background(q_emb, fg0)
            logits0 = scores_to_logits(fg0, bg_score0)
            fg_conf, bg_conf, _ = confident_masks(logits0)
            cur_logits, cur_set = logits0, supp_hyper

            # 3. SSM: Query Self-Supporting Prototype Learning
            if self.flags["ssm"]:
                q_fg = masked_average(q_emb, fg_conf)
                if q_fg is None:
                    q_fg = base_proto
                q_set = self._hyperprototypes(q_emb, q_fg, select_mask=fg_conf)
                fused = self._fuse(supp_hyper, q_set, lam)
                fg1, assign1, _ = gpm_match(q_emb, fused)
                bg_score1 = self._background_score(q_emb, bg_conf, bg_proto0)
                cur_logits = scores_to_logits(fg1, bg_score1)
                cur_set = fused
                fg_conf2, bg_conf2, _ = confident_masks(cur_logits)
                fg_cur, asg_cur = fg1, assign1
            else:
                fg_conf2, bg_conf2 = fg_conf, bg_conf
                fg_cur, asg_cur = fg0, assign0

            # 4. Refine Stage
            if self.flags["refine"]:
                q_fg2 = masked_average(q_emb, fg_conf2)
                if q_fg2 is None:
                    q_fg2 = cur_set[0]
                set2 = self._hyperprototypes(q_emb, q_fg2, select_mask=fg_conf2)
                fused2 = self._fuse(cur_set, set2, lam) if self.flags["ssm"] else set2
                fg2, assign2, _ = gpm_match(q_emb, fused2)
                bg_score2 = self._background_score(q_emb, bg_conf2, bg_proto0)
                cur_logits = scores_to_logits(fg2, bg_score2)
                cur_set = fused2
                fg_cur, asg_cur = fg2, assign2

            final_logits.append(cur_logits)
            init_logits.append(logits0)
            fg_scores.append(fg_cur)
            assigns.append(asg_cur)
            proto_sets.append(cur_set)

        return {
            "logits": torch.stack(final_logits, 0),
            "logits_init": torch.stack(init_logits, 0),
            "fg_score": torch.stack(fg_scores, 0),
            "assign": torch.stack(assigns, 0),
            "embeds": q_embeds,
            "prototypes": proto_sets,
            "feat_size": (fh, fw),
        }

model = OneShotSSMP(modules=MODULES).to(DEVICE)
print("1-Shot SSMP model initialized successfully.")

In [ ]:
# =============================================================================
# 13. TRAINING & VALIDATION HELPERS (CosineAnnealingLR + Grad Clipping)
# =============================================================================
CE_WEIGHTS = torch.tensor([1.0, FG_LOSS_WEIGHT], dtype=torch.float32, device=DEVICE)

def compute_segmentation_loss(out, targets):
    fh, fw = out["feat_size"]
    b = out["logits"].shape[0]
    tgt = F.interpolate(targets.float().unsqueeze(1), size=(fh, fw), mode="nearest").squeeze(1).long()
    logits = out["logits"].permute(0, 2, 1).reshape(b, 2, fh, fw)
    loss = F.cross_entropy(logits, tgt, weight=CE_WEIGHTS)
    if USE_PSEUDO_LOSS:
        init_logits = out["logits_init"].permute(0, 2, 1).reshape(b, 2, fh, fw)
        loss = loss + PSEUDO_LOSS_WEIGHT * F.cross_entropy(init_logits, tgt)
    return loss

def binary_iou(pred, gt):
    inter = torch.logical_and(pred, gt).sum().item()
    union = torch.logical_or(pred, gt).sum().item()
    return inter / max(union, 1)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total, steps = 0.0, 0
    for batch in loader:
        q_img = batch["query_image"].to(DEVICE)
        q_mask = batch["query_mask"].to(DEVICE)
        s_img = batch["supp_image"].to(DEVICE)
        s_mask = batch["supp_mask"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        out = model(q_img, s_img, s_mask)
        loss = compute_segmentation_loss(out, q_mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        total += loss.item()
        steps += 1
    return total / max(steps, 1)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    losses, ious = [], []
    for batch in loader:
        q_img = batch["query_image"].to(DEVICE)
        q_mask = batch["query_mask"].to(DEVICE)
        s_img = batch["supp_image"].to(DEVICE)
        s_mask = batch["supp_mask"].to(DEVICE)

        out = model(q_img, s_img, s_mask)
        losses.append(compute_segmentation_loss(out, q_mask).item())

        fh, fw = out["feat_size"]
        tgt = F.interpolate(q_mask.float().unsqueeze(1), size=(fh, fw), mode="nearest").squeeze(1).long()
        pred = out["logits"].permute(0, 2, 1).reshape(-1, 2, fh, fw).argmax(dim=1)
        for j in range(pred.shape[0]):
            ious.append(binary_iou(pred[j] == 1, tgt[j] == 1))
    return float(np.mean(losses)), float(np.mean(ious))

def fit(model, epochs=EPOCHS):
    history = {"train_loss": [], "val_loss": [], "val_miou": [], "lr": []}
    best_miou = -1.0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        cur_lr = optimizer.param_groups[0]['lr']
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_miou = validate(model, val_loader)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_miou"].append(val_miou)
        history["lr"].append(cur_lr)

        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), BEST_CKPT)

        print(f"Epoch {epoch}/{epochs} | LR: {cur_lr:.6f} | Train Loss: {train_loss:.4f} | Val mIoU: {val_miou:.4f} (best: {best_miou:.4f}, {time.time()-t0:.1f}s)")
    return history

In [ ]:
# =============================================================================
# 14. RUN 1-SHOT TRAINING (Base Classes Only)
# =============================================================================
history = fit(model, epochs=EPOCHS)

In [ ]:
# Restore best model
if BEST_CKPT.exists():
    model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
    print(f"Restored best checkpoint from {BEST_CKPT}")
val_loss, val_miou = validate(model, val_loader)
print(f"Restored Model Validation mIoU: {val_miou * 100:.2f}%")

In [ ]:
# =============================================================================
# 15. 1-SHOT EVALUATION ON NOVEL / UNSEEN CLASSES
# =============================================================================
@torch.no_grad()
def evaluate_oneshot(model, dataset, modules=None):
    old_flags = dict(model.flags)
    if modules is not None:
        model.flags = dict(modules)
    model.eval()

    per_class = {cname: {"tp": 0, "fp": 0, "fn": 0, "tn": 0, "n_images": 0} for cname in UNSEEN_CLASSES}
    total_tp = total_fp = total_fn = total_tn = 0

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    for batch in loader:
        q_img = batch["query_image"].to(DEVICE)
        q_mask = batch["query_mask"].to(DEVICE)
        s_img = batch["supp_image"].to(DEVICE)
        s_mask = batch["supp_mask"].to(DEVICE)
        cnames = batch["target_class_name"]

        out = model(q_img, s_img, s_mask)
        fh, fw = out["feat_size"]
        logits_up = F.interpolate(out["logits"].permute(0, 2, 1).reshape(-1, 2, fh, fw),
                                  size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        preds = (logits_up.argmax(dim=1) == 1)

        for j in range(preds.shape[0]):
            cname = cnames[j]
            p = preds[j]
            gt = (q_mask[j] == 1)

            tp = int((p & gt).sum().item())
            fp = int((p & ~gt).sum().item())
            fn = int((~p & gt).sum().item())
            tn = int((~p & ~gt).sum().item())

            per_class[cname]["tp"] += tp
            per_class[cname]["fp"] += fp
            per_class[cname]["fn"] += fn
            per_class[cname]["tn"] += tn
            per_class[cname]["n_images"] += 1

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_tn += tn

    model.flags = old_flags

    # Compute metrics
    for cname, v in per_class.items():
        denom = max(v["tp"] + v["fp"] + v["fn"], 1)
        v["iou"] = v["tp"] / denom

    valid_ious = [v["iou"] for v in per_class.values() if v["n_images"] > 0]
    miou = float(np.mean(valid_ious)) if valid_ious else 0.0
    fg_iou = total_tp / max(total_tp + total_fp + total_fn, 1)
    bg_iou = total_tn / max(total_tn + total_fp + total_fn, 1)
    fb_iou = 0.5 * (fg_iou + bg_iou)
    pixel_acc = (total_tp + total_tn) / max(total_tp + total_tn + total_fp + total_fn, 1)
    dice = 2 * total_tp / max(2 * total_tp + total_fp + total_fn, 1)

    return {
        "mIoU": miou,
        "fb_iou": fb_iou,
        "pixel_acc": pixel_acc,
        "dice": dice,
        "fg_iou": fg_iou,
        "bg_iou": bg_iou,
        "per_class": per_class,
        "n_images": sum(v["n_images"] for v in per_class.values()),
    }

def print_metrics(results, title="1-SHOT EVALUATION"):
    print("=" * 40)
    print(f"{title}")
    print("=" * 40)
    print(f"Evaluated Images : {results['n_images']}")
    print(f"1-Shot mIoU      : {results['mIoU'] * 100:.2f}%")
    print(f"FB-IoU           : {results['fb_iou'] * 100:.2f}%")
    print(f"Pixel Accuracy   : {results['pixel_acc'] * 100:.2f}%")
    print(f"Dice / F1        : {results['dice'] * 100:.2f}%")
    print("-" * 40)
    print("Per-Class IoU:")
    for cname, v in sorted(results["per_class"].items()):
        print(f"  {cname:<20} : {v['iou'] * 100:6.2f}% ({v['n_images']} images)")
    print("=" * 40)

oneshot_results = evaluate_oneshot(model, test_set)
print_metrics(oneshot_results)

In [ ]:
# =============================================================================
# 16. VISUALIZATIONS (Curves, 1-Shot Segmentation Results, Per-Class Bar Chart)
# =============================================================================
# 1. Training Curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(history["val_loss"], marker="s", label="Val Loss")
axes[0].set_title("1-Shot Training / Validation Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_miou"], marker="o", color="green", label="Val mIoU")
axes[1].set_title("1-Shot Validation mIoU")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mIoU"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 2. Per-Class IoU Bar Chart
names = list(oneshot_results["per_class"].keys())
values = [oneshot_results["per_class"][n]["iou"] * 100 for n in names]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(names, values, color="teal", edgecolor="black")
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)
ax.axhline(oneshot_results["mIoU"] * 100, color="red", ls="--", label=f"mIoU = {oneshot_results['mIoU'] * 100:.2f}%")
ax.set_title("1-Shot Per-Class IoU on Unseen Classes")
ax.set_ylabel("IoU (%)"); ax.legend(); plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

# 3. Qualitative 1-Shot Predictions
def show_oneshot_results(model, dataset, n=3):
    model.eval()
    fig, axes = plt.subplots(n, 5, figsize=(16, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, 5)

    for i in range(min(n, len(dataset))):
        ep = dataset[i]
        s_img = ep["supp_image"].unsqueeze(0).to(DEVICE)
        s_mask = ep["supp_mask"].unsqueeze(0).to(DEVICE)
        q_img = ep["query_image"].unsqueeze(0).to(DEVICE)
        q_mask = ep["query_mask"]
        cname = ep["target_class_name"]

        with torch.no_grad():
            out = model(q_img, s_img, s_mask)
            fh, fw = out["feat_size"]
            pred = (F.interpolate(out["logits"].permute(0, 2, 1).reshape(-1, 2, fh, fw),
                                  size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False).argmax(dim=1)[0].cpu() == 1)

        gt = (q_mask == 1)
        iou = binary_iou(pred, gt)

        raw_q = (denormalize(ep["query_image"]).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        raw_s = (denormalize(ep["supp_image"]).permute(1, 2, 0).numpy() * 255).astype(np.uint8)

        # Overlay: Cyan = TP, Red = FN, Yellow = FP
        ov = raw_q.copy()
        ov[gt & pred] = 0.35 * ov[gt & pred] + 0.65 * np.array([0, 255, 255])
        ov[gt & ~pred] = 0.35 * ov[gt & ~pred] + 0.65 * np.array([255, 0, 0])
        ov[~gt & pred] = 0.35 * ov[~gt & pred] + 0.65 * np.array([255, 255, 0])

        axes[i, 0].imshow(raw_s); axes[i, 0].set_title(f"Support ({cname})")
        axes[i, 1].imshow(ep["supp_mask"], cmap="gray"); axes[i, 1].set_title("Support Mask")
        axes[i, 2].imshow(raw_q); axes[i, 2].set_title("Query Image")
        axes[i, 3].imshow(pred, cmap="gray"); axes[i, 3].set_title(f"1-Shot Pred (IoU {iou*100:.1f}%)")
        axes[i, 4].imshow(ov); axes[i, 4].set_title("Overlay (TP cyan, FN red, FP yellow)")

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout(); plt.show()

show_oneshot_results(model, test_set, n=2)

In [ ]:
# =============================================================================
# 17. 1-SHOT ABLATION STUDY
# =============================================================================
ABLATION_CONFIGS = [
    ("A: Baseline 1-Shot (MAP visual prototype)", dict(hpr=False, ssm=False, asbp=False, refine=False)),
    ("B: + HPR (hyperprototype clustering)",       dict(hpr=True,  ssm=False, asbp=False, refine=False)),
    ("C: + SSM (self-supporting matching)",       dict(hpr=True,  ssm=True,  asbp=False, refine=False)),
    ("D: + ASBP (adaptive background)",           dict(hpr=True,  ssm=True,  asbp=True,  refine=False)),
    ("E: + Refine (full 1-shot SSMP)",            dict(hpr=True,  ssm=True,  asbp=True,  refine=True)),
]

ablation_rows = []
for name, flags in ABLATION_CONFIGS:
    res = evaluate_oneshot(model, test_set, modules=flags)
    ablation_rows.append({
        "Model": name,
        "mIoU (%)": round(res["mIoU"] * 100, 2),
        "FB-IoU (%)": round(res["fb_iou"] * 100, 2),
        "Dice (%)": round(res["dice"] * 100, 2),
        "Pixel Acc (%)": round(res["pixel_acc"] * 100, 2),
    })
    print(f"{name:<45} mIoU = {res['mIoU'] * 100:6.2f}%")

ablation_df = pd.DataFrame(ablation_rows)
print("\n" + "=" * 50)
print("1-SHOT ABLATION SUMMARY")
print("=" * 50)
print(ablation_df.to_string(index=False))

# Restore full model
model.flags = dict(MODULES)

## 18. Comparison with the Original SSMP Paper (1-Shot)

| Module / Setting | Original SSMP (Shen et al. 2024) | This Notebook (1-Shot Recreation) |
|---|---|---|
| **Task** | 1-Shot Semantic Segmentation | **1-Shot** Semantic Segmentation |
| **Support Set** | 1 support image + mask | 1 support image + mask |
| **Prototype Source** | Masked Average Pooling (MAP) | Masked Average Pooling (MAP) + Learnable Adapter |
| **Backbone** | Pretrained ResNet-50 (frozen) | Pretrained ResNet-50 (frozen, layer3+4) |
| **HPR Module** | Superpixel clustering in feature space | K-means clustering in joint feature+spatial space |
| **GPM Module** | Cosine similarity + argmax allocation | Cosine similarity + softmax-weighted guided aggregation |
| **SSM Module** | Pseudo-mask query prototype refinement | Confident query pseudo-labels + lambda fusion |
| **ASBP Module** | Affinity matrix background prototype | Affinity matrix background prototype |
| **Epochs / Optim** | SGD, 20 epochs | Adam + CosineAnnealingLR, 25 epochs |
| **Paper Benchmark** | **39.82% mIoU** on iSAID-5i (1-shot) | Evaluated on iSAID-5i |